# Workflow

To process a document, we intend to the following things:
1. Ingest the data from the file.
2. Break the data into chunks that easily fit into the context window of the model.
3. Create vector embedding for the vector database to retrieve it.
4. Put it to a vector DB.

# Data Ingestion

In [ ]:
import os                                                               # For directory and file creation

In [ ]:
# Libraries that are required for document loading
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader             # For loading tect
from langchain_community.document_loaders import DirectoryLoader        # For loading directories
from langchain_community.document_loaders import PyMuPDFLoader          # For loading PDFs
from langchain_community.document_loaders import PyPDFLoader            # For loading PDFs
from langchain_text_splitters import RecursiveCharacterTextSplitter     # For chunking

In [ ]:
# Libraries required for vector store
import numpy as np                                                      # For storing embeddings as ndarray
import sklearn                                                          # Required by sentence_transformers
import chromadb                                                         # The vector store
import uuid                                                             # For generating unique document IDs
from typing import List, Dict, Any, Tuple                               # For data storing and manipulation. These are non-primitive data types (object of List, Dict, Any and Tuple). Required as loaders return non-primitive datatypes.
from sklearn.metrics.pairwise import cosine_similarity                  # For calculating similarity between query and stored document embeddings
from sentence_transformers import SentenceTransformer                   # Embeddings will be generated from transformers provided by this library

## 1. Data Injestion

Loading all the PDF files from "./data/pdf_files"

In [ ]:
dir_loader = DirectoryLoader(   "./data/pdf_files", 
                                glob="**/*.pdf",
                                loader_cls=PyPDFLoader,
                                show_progress=True
                            )
pdf_documents = dir_loader.load()

## 2. Embedding

Defined the class for Embedding Manager, it's function is to generate embeddings

In [ ]:
class Embedding_Manager:
    # To handle document embedding generation
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
            Initialize embedding manager
            model_name: Huggingface model name for sentence embedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimention {self.get_embedding_dimension()}")
        except Exception as e:
            print(f"Failed loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
            Generate embeddings for a list of texts
            texts: List of text strings to embed
            returns: numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts.")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    def get_embedding_dimension(self) -> int:
        """
            Get the embedding dimension of the model
        """
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()

In [ ]:
embedding_manager = Embedding_Manager()

## 3. Chunking

In [ ]:
def recursive_split_documents(documents,chunk_size=2000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,                              # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap,                        # 200 chars overlap for context
        length_function=len,                                # How to measure length
        separators=["\n", " ", ""]                          # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [ ]:
chunks = recursive_split_documents(pdf_documents,chunk_size=1000 ,chunk_overlap=200)

## 6. Vector Store

In [ ]:
class VectorStore:
    """
        Manages document embeddings in a ChromaDB vector store
    """
    def __init__(self, collection_name: str = "chunks", persist_directory: str = "./data/vector_store"):
        """
            Initialize the vector store
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """
            Initialize ChromaDB client and collection
        """
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection  (   name=self.collection_name, 
                                                                        metadata={"description":"PDF document embeddings for RAG"}
                                                                    )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
            Add documents and their embeddings to vector store
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents not equal to number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store.")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())
            
            # Metadata and content can be used for filtering and retrieval later
            metadatas.append(metadata)
        
        # Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise

In [ ]:
vector_store = VectorStore()
print(f"Type of vector_store: {type(vector_store)}")

## 7. Extracting all texts from the chunks and creating embeddings

In [ ]:
"""
    Convert the text to embeddings
"""
texts = [doc.page_content for doc in chunks]
len(texts)

In [ ]:
"""
    Generate embeddings for the document chunks
"""
embeddings = embedding_manager.generate_embeddings(texts)
len(embeddings)

In [ ]:
"""
    Store in vector store
"""
vector_store.add_documents(chunks,embeddings)

## 8.Retrieval from Vector Store

In [ ]:
class RAGRetriever:
    """
        Handle query-based retrieval from vector store
    """
    def __init__(self, vector_store: VectorStore, embedding_manager: Embedding_Manager):
        """
            Initialize the RAG retriever
            vector_store: Vector store instance
            embedding_manager: Embedding manager instance
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Tuple[str, Dict[str, Any]]]:
        """
            Retrieve relevant documents for a query
            query: User query string
            top_k: Number of top results to return
            score_threshold: Minimum similarity score for retrieval
            returns: List of tuples (document content, metadata)
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance, so similarity = 1 - distance)
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id' : doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents above the similarity threshold.")
            else:
                print(f"No documents retrieved for the query.")
                
            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
    

In [ ]:
rag_retriever = RAGRetriever(vector_store, embedding_manager)
print(f"Type of rag_retriever: {type(rag_retriever)}")

In [ ]:
rag_retriever.retrieve("Event Prediction", top_k=3, score_threshold=0.01)

In [ ]:
rag_retriever.retrieve("what is feature based cycle aware time positional encoding", top_k=3, score_threshold=0.01)